In [2]:
# import library necessary

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.decomposition import PCA
from sklearn.decomposition import TruncatedSVD
import nltk
from wordcloud import WordCloud
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.decomposition import LatentDirichletAllocation

In [4]:
# read in songs and replace unwanted characters (new line)

df = pd.read_excel("C:\\Users\\USER\\Documents\\misc\\Certificate_agency.xlsx" ,
                                sheet_name = 'Main sheet',
                                header=0)
df['CSR_CERTIFIED_BY'] = df['CSR_CERTIFIED_BY'].str.replace('\n', '')
df.head()

In [5]:
# components for features reduction
n_components = 5

# number of clusters we want
n_clusters = 5

# covert words into TFIDF metrics
tfidf = TfidfVectorizer(stop_words = 'english')
X_text = tfidf.fit_transform(df['CSR_CERTIFIED_BY'])

# reduce dimensions
svd = TruncatedSVD(n_components=n_components, random_state = 0)
X_2d = svd.fit_transform(X_text)

# fit k-mean clustering
kmeans = KMeans(n_clusters=n_clusters, random_state = 0)

# predict our clusters for each song
X_clustered = kmeans.fit_predict(X_2d)

# display by groups
df_plot = pd.DataFrame(list(X_2d), list(X_clustered))
df_plot = df_plot.reset_index()
df_plot.rename(columns = {'index': 'Cluster'}, inplace = True)
df_plot['Cluster'] = df_plot['Cluster'].astype(int)

print(df_plot.head())

print(df_plot.groupby('Cluster').agg({'Cluster': 'count'}))

In [8]:
# Logistic Regression approach

df['Cluster'] = df_plot['Cluster']

# function for finding most significant words for each cluster
def generate_text(cluster):
    
    df_s = df['CSR_CERTIFIED_BY']
    y = df['Cluster'].map(lambda x: 1 if x == cluster else 0)
    count = len(df_s)
    
    tfidf = TfidfVectorizer(stop_words = 'english')
    X = tfidf.fit_transform(df_s)
    X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)
    clf = LogisticRegression(random_state = 0).fit(X_train, y_train)
    clf_d = DummyClassifier().fit(X_train, y_train)
    acc = clf.score(X_test, y_test)
    acc_d = clf_d.score(X_test, y_test)
    coef = clf.coef_.tolist()[0]
    w = tfidf.get_feature_names()
    coeff_df = pd.DataFrame({'words' : w, 'score' : coef})
    coeff_df = coeff_df.sort_values(['score', 'words'], ascending=[0, 1])
    coeff_df = coeff_df[:30]
    d = coeff_df.set_index('words')['score'].to_dict()
    return d, acc, acc_d

In [13]:
# visualized it by word clouds
fig, ax = plt.subplots(n_clusters, sharex=True, figsize=(15,10*n_clusters))

for i in range(0, n_clusters):
    d, acc, acc_d = generate_text(i)
    wordcloud = WordCloud(max_font_size=40, collocations=False, colormap = 'Reds', background_color = 'white').fit_words(d)
    ax[i].imshow(wordcloud, interpolation='bilinear')
    ax[i].set_title('Cluster {} \nLR accuracy: {} \nDummy classifier accuracy: {}'.format(i, acc, acc_d), fontsize = 20)
    ax[i].axis("off")

In [11]:
df['Cluster'].unique()

In [12]:
# LDA
no_topics = 5

c = CountVectorizer(stop_words='english')
X_text_c = c.fit_transform(df['CSR_CERTIFIED_BY'])

lda = LatentDirichletAllocation(learning_method = 'online', n_components=no_topics, random_state=0).fit(X_text_c)
X_text_c_feature_names = c.get_feature_names()

def display_topics(model, feature_names, no_top_words):
    for topic_idx, topic in enumerate(model.components_):
        print ("Topic %d:" % (topic_idx))
        print (" ".join([feature_names[i] for i in topic.argsort()[:-no_top_words - 1:-1]]))

no_top_words = 10
display_topics(lda, X_text_c_feature_names, no_top_words)

In [19]:
# read in songs and replace unwanted characters (new line)

df = pd.read_excel("C:\\Users\\USER\\Documents\\misc\\all_hotels_cert.xlsx" ,
                                sheet_name = 'Clean_Results',
                                header=0)
df['CSR_CERTIFIED_BY'] = df['CSR_CERTIFIED_BY'].str.replace('\n', '')
df.head()

In [21]:
import jellyfish as jelly

list1 = list(df['CSR_CERTIFIED_BY'] )
list2 = list(df['CSR_CERTIFIED_BY'] )

list3 = []
empty_tuple = ()
for i in range(0, len(list1)):
    for j in range(0, len(list2)):
        score = jelly.jaro_winkler(list1[i], list2[j])
        if score > 0.50 and score < 1.0:
            empty_tuple = (list1[i], list2[j], score)
            list3.append(empty_tuple)
list3[0:10]

In [22]:
import pandas as pd
df = pd.DataFrame(list3, columns = ['Name' , 'Matched_name', 'Match_score']) 

In [28]:
df.groupby(['Name']).count()